In [ ]:
from pathlib import Path
from typing import Literal
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from common import load_data, ggcms, ssps
import string

In [ ]:
def format_feature_name(name: str) -> str:
    return {
        "TMXav": "Avg. max. temperature [°C]", 
        "PRCPsum": "Avg. precipitation [mm/day]", 
        "RADsum": "Avg. radiation [MJ/m²/day]", 
        "LEN": "GS length", 
        "KDD": "Killing degree days [% of GS]", 
        "PETsum": "Avg. PET [mm/day]",
        "CMDgt0": "Days where CMD > 0 [% of GS]"
    }[name]

In [ ]:
features_plot = ["TMXav", "PRCPsum", "RADsum", "KDD", "CMDgt0", "PETsum"]
ggcms_plot = [x for x in ggcms if x != "GSWP3"]
ssps_plot = ["ssp245", "ssp370"]
labels = string.ascii_lowercase

periods = [
    (2015, 2045),
    (2070, 2099),
]

periods_hist = [
    (1971, 1985),
    (1985, 2014),
]

hist_from = 2000
hist_to = 2014

colors = list(mcolors.TABLEAU_COLORS)

plt.rcParams.update({"font.size": 14})

fig, axs = plt.subplots(
    len(features_plot),
    len(ssps_plot),
    figsize=(15, 20),
    sharey="row",
    layout="compressed",
)

for row, feature in enumerate(features_plot):

    for col, ssp in enumerate(ssps_plot):

        ax = axs[row, col]

        year_from, year_to = periods[1]

        data = []
        data_hist = []

        ############################################################
        # Load all data first
        ############################################################

        for ggcm in ggcms_plot:

            hist = (
                load_data("historical", ggcm, "mai")
                .filter(
                    pl.col("YR").is_between(hist_from, hist_to),
                    pl.col("PERIOD") == "gs",
                )
                .sample(100_000)
                .select([feature, "LEN"])
            )

            future = (
                load_data(ssp, ggcm, "mai")
                .filter(
                    pl.col("YR").is_between(year_from, year_to),
                    pl.col("PERIOD") == "gs",
                )
                .sample(100_000)
                .select([feature, "LEN"])
            )

            if feature in ["PRCPsum", "RADsum", "PETsum"]:
                hist = hist.with_columns(
                    (pl.col(feature) / pl.col("LEN")).alias(feature)
                )
                future = future.with_columns(
                    (pl.col(feature) / pl.col("LEN")).alias(feature)
                )

            elif feature in ["KDD", "CMDgt0"]:
                hist = hist.with_columns(
                    (pl.col(feature) / pl.col("LEN") * 100).alias(feature)
                )
                future = future.with_columns(
                    (pl.col(feature) / pl.col("LEN") * 100).alias(feature)
                )

            data_hist.append(hist[feature].to_numpy())
            data.append(future[feature].to_numpy())

        ############################################################
        # Plot
        ############################################################

        vp = ax.violinplot(data, showextrema=False)
        vp_hist = ax.violinplot(data_hist, showextrema=False)

        for body, color in zip(vp["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_alpha(0.6)

        for body in vp_hist["bodies"]:
            body.set_facecolor("none")
            body.set_edgecolor("black")
            body.set_linewidth(1.0)
            body.set_linestyle("--")
            body.set_alpha(0.6)

        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.set_xticks([])

        if feature == "PRCPsum":
            vals = np.concatenate(data + data_hist)
            lo = np.percentile(vals, 1)
            hi = np.percentile(vals, 99)
            ax.set_ylim(lo, hi)
        
        elif feature == "KDD":
            vals = np.concatenate(data + data_hist)
            hi = np.percentile(vals, 95)
            ax.set_ylim(0, hi)

        if col == 0:
            ax.set_ylabel(format_feature_name(feature))

        if row == 0:
            ax.set_title(f"{ssp.upper()} ({year_from}–{year_to})")

        # Panel label
        panel = row * len(ssps_plot) + col
        ax.text(
            0.02,
            0.95,
            labels[panel],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=16,
            fontweight="bold",
        )

############################################################
# Legend
############################################################

handles = [
    Patch(facecolor=color, label=label)
    for color, label in zip(colors, ggcms_plot)
]

fig.legend(
    handles=handles,
    labels=ggcms_plot,
    ncol=len(ggcms_plot),
    loc="upper center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    fontsize=14,
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig("features_violins.pdf")
plt.show()